In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**Datasets:**

'/kaggle/input/competitions/rsna-knee-abnormality-detection'
'/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series'
'/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series'
'/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv'
'/kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv'
'/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv'
'/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv'
'/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv'

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

# 1. Tạo danh sách chứa các đường dẫn file
file_paths = []

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        file_paths.append(os.path.join(dirname, filename))

# 2. Tạo DataFrame từ danh sách và xuất thành file CSV
df_files = pd.DataFrame(file_paths, columns=['file_path'])
df_files.to_csv('file_list.csv', index=False)

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**Link Datasets:**

**The files containing the links are located within this file:**
(Nằm trong file chứa các link:)

'/kaggle/input/datasets/trnhquangminh140/rsna-knee-abnormality-detection-link-datasets/file_list_dataset.txt'

**And the links within the file are similar to the following:**
(và các link bên trong file tương tự như sau:) 

'/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv',
'/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv',
'/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv',
'/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv',
'/kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv',

'/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series/1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776/1.2.826.0.1.3680043.8.498.10818107180155454609278933308651876447/1.2.826.0.1.3680043.8.498.43758349334868121588037306541272735960.dcm'

'/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series/1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776/1.2.826.0.1.3680043.8.498.10818107180155454609278933308651876447/1.2.826.0.1.3680043.8.498.98300178100494838040275925015059424504.dcm'

'/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series/1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776/1.2.826.0.1.3680043.8.498.10818107180155454609278933308651876447/1.2.826.0.1.3680043.8.498.24211175366460976919052329259009027034.dcm'

'/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series/1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776/1.2.826.0.1.3680043.8.498.10818107180155454609278933308651876447/1.2.826.0.1.3680043.8.498.62702218416158351740972867662762507972.dcm'



**And there are many links, including files; the dataset file_list_dataset.txt has a size of up to 211MB.**
(và nhiều link lắm, file chứ link dataset file_list_dataset.txt có dung lượng đến 211MB.)



In [ ]:
# ==============================================================================
# RSNA KNEE ABNORMALITY DETECTION - FULL WORKING END-TO-END PIPELINE
# Author: Trịnh Quang Minh (trnhquangminh140)
# ==============================================================================

import os
import sys
import gc
import re
import glob
import math
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Visualizations / Hiển thị hình ảnh
import cv2

# PyTorch & Medical Imaging Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

# Tải tự động các thư viện cần thiết nếu chưa có
try:
    import pydicom
except ImportError:
    os.system("pip install -q pydicom timm albumentations")
    import pydicom

try:
    import timm
except ImportError:
    os.system("pip install -q timm")
    import timm

warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CONFIGURATION & REPRODUCIBILITY / CẤU HÌNH & TÍNH TÁI LẬP
# ------------------------------------------------------------------------------
class Config:
    SEED = 42
    
    # Đường dẫn thư mục dữ liệu trên Kaggle
    DATA_DIR = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
    OUTPUT_DIR = '/kaggle/working/models'
    
    # Thông số ảnh & thể tích MRI
    IMG_SIZE = 256
    NUM_SLICES_PER_SERIES = 16  # Số lát cắt chuẩn hóa cho mỗi chuỗi MRI (2.5D Input)
    
    # Thông số huấn luyện
    BATCH_SIZE = 8
    NUM_WORKERS = 2
    EPOCHS = 3
    LR = 3e-4
    WEIGHT_DECAY = 1e-2
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 12 Nhãn chẩn đoán bất thường khớp gối
    LABELS = [
        'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
        'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
        'Synovitis', "Baker's", 'Contusion', 'Fracture'
    ]

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(Config.SEED)
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

print(f"=== Running on Device / Đang chạy trên thiết bị: {Config.DEVICE} ===")
if Config.DEVICE.type == 'cpu':
    print("⚠️ LƯU Ý: Bạn đang chạy trên CPU! Hãy bật GPU trong 'Session options' trên Kaggle để tăng tốc.")

# ------------------------------------------------------------------------------
# 2. DICOM LOADER / TẢI VÀ TIỀN XỬ LÝ ẢNH DICOM
# ------------------------------------------------------------------------------
def load_dicom_array(fpath, target_size=Config.IMG_SIZE):
    """
    Đọc file DICOM an toàn, thực hiện Rescale & Normalization về [0, 1]
    """
    try:
        dcm = pydicom.dcmread(fpath)
        img = dcm.pixel_array.astype(np.float32)
        
        # Rescale Slope & Intercept
        slope = getattr(dcm, 'RescaleSlope', 1)
        intercept = getattr(dcm, 'RescaleIntercept', 0)
        img = img * slope + intercept
        
        # Min-Max Normalization
        img_min, img_max = img.min(), img.max()
        if img_max > img_min:
            img = (img - img_min) / (img_max - img_min)
        else:
            img = np.zeros_like(img)
            
        img = cv2.resize(img, (target_size, target_size))
        return img
    except Exception as e:
        return np.zeros((target_size, target_size), dtype=np.float32)

# ------------------------------------------------------------------------------
# 3. DATASET & DATALOADER DEFINITION / ĐỊNH NGHĨA DATASET
# ------------------------------------------------------------------------------
class RSNAKneeDataset(Dataset):
    def __init__(self, df, series_df, base_path, is_train=True):
        self.df = df
        self.series_df = series_df
        self.base_path = base_path
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_uid = row['StudyInstanceUID']
        
        # Tìm tất cả chuỗi series thuộc ca bệnh (StudyInstanceUID)
        study_series = self.series_df[self.series_df['StudyInstanceUID'] == study_uid] if self.series_df is not None else pd.DataFrame()
        
        volume_slices = []
        folder_prefix = 'train_series' if self.is_train else 'test_series'
        
        if not study_series.empty:
            for _, s_row in study_series.iterrows():
                series_uid = s_row['SeriesInstanceUID']
                series_dir = os.path.join(self.base_path, folder_prefix, study_uid, series_uid)
                if os.path.exists(series_dir):
                    dcm_files = sorted(glob.glob(os.path.join(series_dir, "*.dcm")))
                    if len(dcm_files) > 0:
                        # Lấy mẫu đều các lát cắt trong chuỗi
                        step = max(1, len(dcm_files) // 4)
                        selected = dcm_files[::step][:4]
                        for f in selected:
                            volume_slices.append(load_dicom_array(f))
        
        # Padding hoặc Clipping để đưa về đúng số lượng kênh C=16 (2.5D Input)
        target_depth = Config.NUM_SLICES_PER_SERIES
        if len(volume_slices) == 0:
            volume_slices = [np.zeros((Config.IMG_SIZE, Config.IMG_SIZE), dtype=np.float32) for _ in range(target_depth)]
        elif len(volume_slices) < target_depth:
            pad_count = target_depth - len(volume_slices)
            volume_slices.extend([volume_slices[-1]] * pad_count)
        else:
            volume_slices = volume_slices[:target_depth]
            
        # Chuyển đổi thành Tensor 3D (C, H, W)
        volume_tensor = torch.tensor(np.stack(volume_slices), dtype=torch.float32)
        
        if self.is_train:
            labels = row[Config.LABELS].values.astype(np.float32)
            return volume_tensor, torch.tensor(labels, dtype=torch.float32), study_uid
        else:
            return volume_tensor, study_uid

# ------------------------------------------------------------------------------
# 4. NEURAL NETWORK ARCHITECTURE / MÔ HÌNH PRETRAINED BACKBONE
# ------------------------------------------------------------------------------
class RSNAKneeModel(nn.Module):
    def __init__(self, num_classes=12, in_channels=16, backbone_name='resnet34'):
        super(RSNAKneeModel, self).__init__()
        
        # Lớp chuyển đổi đầu vào 16 kênh (slices) về 3 kênh chuẩn của ImageNet
        self.stem = nn.Conv2d(in_channels, 3, kernel_size=1, bias=False)
        
        # Tải mô hình Pretrained từ timm
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        in_features = self.backbone.num_features
        
        # Classifier Head cho 12 nhãn
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits

# ------------------------------------------------------------------------------
# 5. TRAINING & VALIDATION LOOPS / HÀM HUẤN LUYỆN
# ------------------------------------------------------------------------------
def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    for images, labels, _ in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with autocast(enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
    return running_loss / len(dataloader.dataset)

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels, _ in tqdm(dataloader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
    return running_loss / len(dataloader.dataset)

# ------------------------------------------------------------------------------
# 6. MAIN PIPELINE / QUY TRÌNH CHẠY CHÍNH
# ------------------------------------------------------------------------------
def run_pipeline():
    print("\n--- [Step 1] Loading Metadata / Tải dữ liệu metadata ---")
    train_csv_path = os.path.join(Config.DATA_DIR, 'train.csv')
    train_series_csv_path = os.path.join(Config.DATA_DIR, 'train_series.csv')
    test_csv_path = os.path.join(Config.DATA_DIR, 'test.csv')
    test_series_csv_path = os.path.join(Config.DATA_DIR, 'test_series.csv')
    
    # Kiểm tra sự tồn tại của dữ liệu
    if os.path.exists(train_csv_path):
        train_df = pd.read_csv(train_csv_path)
        train_series_df = pd.read_csv(train_series_csv_path) if os.path.exists(train_series_csv_path) else None
        print(f"Loaded train.csv with {len(train_df)} samples.")
    else:
        print("⚠️ Warning: Không tìm thấy train.csv! Khởi tạo dữ liệu mẫu để kiểm tra pipeline.")
        train_df = pd.DataFrame({'StudyInstanceUID': ['sample_1', 'sample_2']})
        for lbl in Config.LABELS:
            train_df[lbl] = 0.0
        train_series_df = None

    if os.path.exists(test_csv_path):
        test_df = pd.read_csv(test_csv_path)
        test_series_df = pd.read_csv(test_series_csv_path) if os.path.exists(test_series_csv_path) else None
        print(f"Loaded test.csv with {len(test_df)} samples.")
    else:
        test_df = pd.DataFrame({'StudyInstanceUID': ['sample_test_1', 'sample_test_2']})
        test_series_df = None

    # Tạo DataLoader cho Training
    train_dataset = RSNAKneeDataset(train_df, train_series_df, base_path=Config.DATA_DIR, is_train=True)
    train_loader = DataLoader(
        train_dataset, 
        batch_size=Config.BATCH_SIZE, 
        shuffle=True, 
        num_workers=Config.NUM_WORKERS,
        pin_memory=True if Config.DEVICE.type == 'cuda' else False
    )

    # Khởi tạo Model, Loss, Optimizer
    model = RSNAKneeModel(num_classes=len(Config.LABELS), in_channels=Config.NUM_SLICES_PER_SERIES)
    model.to(Config.DEVICE)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    scaler = GradScaler(enabled=(Config.DEVICE.type == 'cuda'))

    print("\n--- [Step 2] Training Model / Tiến hành huấn luyện ---")
    for epoch in range(Config.EPOCHS):
        print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, Config.DEVICE)
        print(f"Epoch {epoch+1} - Train Loss: {train_loss:.4f}")

    # Lưu checkpoint
    model_save_path = os.path.join(Config.OUTPUT_DIR, 'rsna_knee_model.pth')
    torch.save(model.state_dict(), model_save_path)
    print(f"\nModel checkpoint saved at: {model_save_path}")

    # Tiến hành suy luận tập Test (Inference)
    print("\n--- [Step 3] Generating Test Predictions / Dự đoán tập Test ---")
    test_dataset = RSNAKneeDataset(test_df, test_series_df, base_path=Config.DATA_DIR, is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    
    model.eval()
    results = []
    
    with torch.no_grad():
        for images, uids in tqdm(test_loader, desc="Inference"):
            images = images.to(Config.DEVICE)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            
            for uid, prob in zip(uids, probs):
                res_row = {'StudyInstanceUID': uid}
                for idx, label in enumerate(Config.LABELS):
                    res_row[label] = float(np.round(prob[idx], 4))
                results.append(res_row)

    sub_df = pd.DataFrame(results)
    submission_path = 'submission.csv'
    sub_df.to_csv(submission_path, index=False)
    
    print(f"\n✅ Đã lưu tệp submission.csv thành công!")
    print("\n5 dòng đầu tiên của submission.csv:")
    print(sub_df.head())

if __name__ == '__main__':
    run_pipeline()